# Data acquisition: practice

Nine short exercises using real [UCI Bike Sharing data](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). Each row in `day.csv` represents one day of bike rentals. `cnt` is the total; `casual` and `registered` are its parts. `workingday` is 1 for a working day and 0 otherwise.

Source: Fanaee-T (2013), *Bike Sharing*, UCI. [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Exported files use the same real records.

## Setup

Run this cell first. Practice files will be saved in `bike_practice`.

In [2]:
import requests
import zipfile
from io import BytesIO
from pathlib import Path
import pandas as pd
from IPython.display import display

url = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
folder = Path('bike_practice')
folder.mkdir(exist_ok=True)

## 1. Make an HTTP request

Download the ZIP using `requests.get` with a timeout. Print the HTTP status, content type and number of downloaded bytes. Check for HTTP errors, list the ZIP contents, and load `day.csv` into `df`.

Try it below, then check the answer.

In [3]:
response = requests.get(url, timeout = 30)
print(f"Status Code: {response.status_code}")
print(f"Content Type: {response.headers.get('Content-Type')}")
print(f"Number of bytes: {len(response.content)}")


with zipfile.ZipFile(BytesIO(response.content)) as archive:
    print(f"Files: {archive.namelist()}")
    with archive.open('day.csv') as csv_file:
        df = pd.read_csv(csv_file)

# response.content:
# is the raw binary payload(a bytes object) returned from an HTTP request
#Normally python's zipfule module expects a file path on disk(like "data.zip")
#or a file-like object that supports methods like .read() or .seek()

#Normally you have response.content which is the untouched binary data,
#rresponse.text which is string decoded using the detected encoding, usually UTF-8
#and then you have response.json() which is the structured object parsed from the text string

#io.Bytes()
# wraps those raw bytes in an in-memory buffer, making them behave exactly
#like an open file on your hard drive - avoiding the overhead of saving a temporary file to disk

#with zipfile.ZipFile(...) as archive:
#zipfile.ZipFile(...) initializes the ZIP reader around that in-memory byte stream
#The with statement creates a context manager. Once the code exists this intended block
#(or if an error is thrown), it automatically closes the archive handle and releases associated resources

#You might be askign why we dont do this in normal http requests:
#Well, when you call response.text or await res.json(), the HTTP library inspects the Content-Type and Content Encoding headers and 
#does the heavy lifting.

#So why do we have to do this now?
#Zip files are not text: A Zip file contains binary compressed data, checksums and non-UTF-8 header flags.
#Trying to run .decode("utf-8") on raw ZIP bytes throws an error because many byte sequences are invalid in text encoding

#While an HTTP library knows how to parse commoon text formats(like JSON or HTML), parsing
#parsing application-specific binary files like ZIPs, Pngs, PDFs, audio requires specialised parsers like zipfile

#archive.namelist()
#inspets the ZIP file's central directory and returns a python list of strings representing every file and 
#directory path archived inside

#with archive.open('day.csv') as csv_file: Reaches inside the ZIP and opens just the file named day.csv
#df = pd.read_csv(csv_file) hands that opened csv to pandas, which reads the rows and columns into a dataframe

#Here is the step-by-step breakdown:

# BytesIO(response.content): Grabs the raw ZIP data downloaded from the web and treats it like an open file sitting directly in your computer's RAM.

# zipfile.ZipFile(...) as archive:: Unlocks and opens the virtual ZIP folder in memory.

# print(f"Files: {archive.namelist()}"): Prints out a checklist of all file names packed inside that ZIP folder.

# with archive.open('day.csv') as csv_file:: Reaches inside the ZIP and opens just the file named day.csv.

# df = pd.read_csv(csv_file): Hands that opened CSV to Pandas, which reads the rows and columns into a DataFrame (df), making it instantly ready to filter, graph, or analyze.

#Without with:
# file = open("data.txt", "r")
# try:
#     content = file.read()
#     # If something crashes here, file.close() never runs!
# finally:
#     file.close()  # You must remember to do this

# With "with":
# with open("data.txt", "r") as file:
#     content = file.read()
# # The file is 100% guaranteed to be closed here!

#With "with": You never have to write archive.close() or csv_file.close()


Status Code: 200
Content Type: None
Number of bytes: 279992
Files: ['Readme.txt', 'day.csv', 'hour.csv']


## 2. Inspect and check the data

Show the shape, column information, first five and last five rows. Count missing values per column and duplicate rows. Is a successful download enough to trust the import?

Try it below, then check the answer.

In [4]:
print(f"Shape: {df.shape}")
df.info()



Shape: (731, 16)
<class 'pandas.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    str    
 2   season      731 non-null    int64  
 3   yr          731 non-null    int64  
 4   mnth        731 non-null    int64  
 5   holiday     731 non-null    int64  
 6   weekday     731 non-null    int64  
 7   workingday  731 non-null    int64  
 8   weathersit  731 non-null    int64  
 9   temp        731 non-null    float64
 10  atemp       731 non-null    float64
 11  hum         731 non-null    float64
 12  windspeed   731 non-null    float64
 13  casual      731 non-null    int64  
 14  registered  731 non-null    int64  
 15  cnt         731 non-null    int64  
dtypes: float64(4), int64(11), str(1)
memory usage: 91.5 KB


In [5]:
df.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [6]:
df.tail()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
726,727,2012-12-27,1,1,12,0,4,1,2,0.254167,0.226642,0.652917,0.350133,247,1867,2114
727,728,2012-12-28,1,1,12,0,5,1,2,0.253333,0.255046,0.590000,0.155471,644,2451,3095
728,729,2012-12-29,1,1,12,0,6,0,2,0.253333,0.242400,0.752917,0.124383,159,1182,1341
729,730,2012-12-30,1,1,12,0,0,0,1,0.255833,0.231700,0.483333,0.350754,364,1432,1796
730,731,2012-12-31,1,1,12,0,1,1,2,0.215833,0.223487,0.577500,0.154846,439,2290,2729


## 3. Semicolon CSV

Save `df` as `bikes.csv` using `;` and no index. Read it back correctly. What happens if you omit `sep`?

Try it below, then check the answer.

In [7]:
path = folder / 'bikes.csv'#bike_practice/bikes.csv
df.to_csv(path, sep=';', index=False)

#The above exports the dataframe to the locationd defined by path
#sep=';' uses semicolons instead of default commass to seperate values.
#index=False omits the row numbers(0,1,2) from the file. Without this flag,
#pandas saves an unnamed extra column containing the row indices

csv_df = pd.read_csv(path, sep=';')
#Reads the saved file back from disk into a fresh dataframe named csv_df
#Specifying sep=';' tells pandas to split columns on semicolons. If you omit this,
#pandas defaults to commas, failing to parse the columns and reading the entire row into a single column
csv_df.head()


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [8]:
csv_df = pd.read_csv(path)
csv_df.head()

,instant;dteday;season;yr;mnth;holiday;weekday;workingday;weathersit;temp;atemp;hum;windspeed;casual;registered;cnt
0,1;2011-01-01;1;0;1;0;6;0;2;0.344167;0.363625;0...
1,2;2011-01-02;1;0;1;0;0;0;2;0.363478;0.353739;0...
2,3;2011-01-03;1;0;1;0;1;1;1;0.196364;0.189405;0...
3,4;2011-01-04;1;0;1;0;2;1;1;0.2;0.212122;0.5904...
4,5;2011-01-05;1;0;1;0;3;1;1;0.226957;0.22927;0....


## 4. Select columns from TXT

Save `df` as tab-separated `bikes.txt`, without an index. Read only `dteday` and `cnt`, parsing the date column. Show the first five rows and the date range.

Try it below, then check the answer.

In [9]:
path = folder / 'bikes.txt'
df.to_csv(path, sep='\t', index=False)
#.to_csv() doesnt care about the extension. It simply takes a table
#and writes it out as delimited text lines

#You could name the file 'bikes.data' or 'bikes.anything' and .to_csv() would still work
txt_df = pd.read_csv(path, sep='\t',
                     usecols=['dteday', 'cnt'],
                     parse_dates=['dteday'])
#parse_dates inspects the values and converts the value from a generic text string
#to an actual atetime64 object
#Once converted you can perform date operatiosn such as extracting the month or day of the week
#(df['dteday'].dt.day_name())

txt_df.head()



,dteday,cnt
0,2011-01-01,985
1,2011-01-02,801
2,2011-01-03,1349
3,2011-01-04,1562
4,2011-01-05,1600


In [10]:
print(f"Date range is from {txt_df['dteday'].min()} to {txt_df['dteday'].max()}")

Date range is from 2011-01-01 00:00:00 to 2012-12-31 00:00:00


## 5. Filter JSON records

Save `df` as `bikes.json` using `orient="records"`, then read it back. Select days with more than 5,000 rentals. Show their dates and counts, and print how many days matched.

Try it below, then check the answer.

In [ ]:
path = folder / 'bikes.json'
df.to_json(path, orient='records')
""" 
orient: options:
records: List of dictionaries -> [{"dteday":"...","cnt":985}, ...]
columns(default): dict of column arrays: {"dteday":{"0":"...","1":"..."}, "cnt":{"0":985,"1":801}}
split: {"columns":["dteday","cnt"],"data":[["...",985], ...]}
values: [["2011-01-01", 985], ["2011-01-02", 801]]
index: {"0":{"dteday":"...","cnt":985}, "1":{...}}
"""
json_df = pd.read_json(path, orient='records', convert_dates=False)
#convert_dates=True: datatype = datetime64
#convert_dates=False: datatype = object(string)
busy_days=json_df.loc[json_df['cnt']>5000, ['dteday', 'cnt']]

""" 
for these exercises we dont really need json_df for example. We can use df.
Tutorials do this to teach you how to save into json and load from json
"""
print(f"Number of busy days is: {len(busy_days)}")


Number of busy days is: 286


## 6. Filter and sort

Find the five working days with the highest rental counts. Show only `dteday`, `temp` and `cnt`. Save the result as `top_working_days.csv`, without an index.

Try it below, then check the answer.

In [12]:
path = folder / 'top_working_days.csv'
df.to_csv(path, index=False)
df = pd.read_csv(path)

top_5 = df.loc[df['workingday'] == 1, ['dteday', 'temp', 'cnt']].sort_values(by='cnt', ascending=False).head(5)
top_5


,dteday,temp,cnt
447,2012-03-23,0.601667,8362
571,2012-07-25,0.724167,8173
629,2012-09-21,0.599167,8167
643,2012-10-05,0.615000,8156
622,2012-09-14,0.633333,8009


## 7. Compare groups

Compare the mean, minimum and maximum daily rentals for working days and non-working days. Which group has the higher average?

Try it below, then check the answer.

In [13]:
comparison = df.groupby('workingday')['cnt'].agg(
    mean='mean',
    min='min',
    max='max'
)
comparison

,mean,min,max
workingday,,,
0,4330.168831,605,8714
1,4584.820000,22,8362


## 8. Calculate and validate

Create `casual_pct`: casual rentals as a percentage of total rentals. Show its first five values. Check whether `casual + registered` equals `cnt` for every day.

Try it below, then check the answer.

In [ ]:
df['casual_pct'] = (df['casual'] / df['cnt']) * 100
print(df['casual_pct'].head())
is_sum_exact = (df['casual'] + df['registered'] == df['cnt']).all()
print(is_sum_exact)
#If you didnt do .all(), you would see True printed for the amount of records in the dataset
#.all() just returns 1 True

0    33.604061
1    16.354557
2     8.895478
3     6.914213
4     5.125000
Name: casual_pct, dtype: float64
True


## 9. Work with dates

Convert `dteday` to dates. Calculate total rentals for each year-month, keeping 2011 and 2012 separate. Show the three busiest months.

Try it below, then check the answer.

In [ ]:
df['dteday'] = pd.to_datetime(df['dteday'])
monthly_rentals = df.groupby(df['dteday'].dt.to_period('M'))['cnt'].sum().sort_values(ascending=False)
""" 
dteday(original date)       df['dteday'].dt.to_period('M')
2011-01-01                  2011-01
2011-01-15                  2011-01 
2011-02-01                  2011-02

Other examples:
'W' (Week): 2011-05-15 -> 2011-05-09/2011-05-15
'D' (Day): 2011-05-15 14:30:00 -> 2011-05-15
"""
monthly_rentals.head(3)


dteday
2012-09    218573
2012-08    214503
2012-07    203607
Freq: M, Name: cnt, dtype: int64